In [1]:
import ee
import folium
import os
from datetime import datetime, timedelta

# --- Configuration Constants ---
PROJECT_ID = "wrkfarm-415118"
SERVICE_ACCOUNT_EMAIL = "wrkfarm-415118@appspot.gserviceaccount.com"
SERVICE_ACCOUNT_KEY_FILE = "wrkfarm-415118-8bd4bb22e26c.json" # Your downloaded key file

# Polygon from your notebook
TEST_POLYGON_COORDS = [[
    [77.77333199305133, 12.392392446684909],
    [77.77285377084087, 12.391034719901086],
    [77.77415744218291, 12.390603704636632],
    [77.77438732135664, 12.391302225016886],
    [77.77376792469431, 12.391501801924363],
    [77.77399141833513, 12.392187846379386],
    [77.77333199305133, 12.392392446684909]
]]
# -------------------------------

# --- 1. Authentication ---
print("Authenticating with Google Earth Engine...")
try:
    if not os.path.exists(SERVICE_ACCOUNT_KEY_FILE):
        raise FileNotFoundError(
            f"Key file not found at '{SERVICE_ACCOUNT_KEY_FILE}'."
        )
    
    credentials = ee.ServiceAccountCredentials(
        SERVICE_ACCOUNT_EMAIL,
        SERVICE_ACCOUNT_KEY_FILE
    )
    ee.Initialize(credentials=credentials, project=PROJECT_ID)
    print(f"✅ Authentication successful for project: {PROJECT_ID}")

except Exception as e:
    print(f"❌ Authentication failed: {e}")
    # Stop execution if auth fails
    raise

# --- 2. Cloud Masking Function ---
# Function to mask clouds using the Sentinel-2 SCL (Scene Classification Layer) band
def maskS2clouds(image):
    scl = image.select('SCL')
    
    # We want to keep pixels that are NOT:
    # 3 (Cloud Shadow)
    # 8 (Medium Probability Cloud)
    # 9 (High Probability Cloud)
    # 10 (Cirrus)
    
    # Create a mask for all "bad" pixels by combining them with .Or()
    bad_pixels = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    
    # Invert the mask to get "good" pixels (where bad_pixels is 0)
    mask = bad_pixels.Not()
    
    # Scale and offset factors are applied to optical bands
    return image.updateMask(mask).divide(10000) \
                .select("B.*") \
                .copyProperties(image, ["system:time_start"])

# --- 3. NDVI Calculation ---
print("Calculating NDVI...")
try:
    # Define the Area of Interest (AOI)
    aoi = ee.Geometry.Polygon(TEST_POLYGON_COORDS)

    # Get the most recent 6 months of data
    start_date = (datetime.now() - timedelta(days=180)).strftime('%Y-%m-%d')
    end_date = datetime.now().strftime('%Y-%m-%d')

    # Find the least cloudy image in the last 6 months
    collection = ee.ImageCollection('COPERNICUS/S2_SR') \
                   .filterDate(start_date, end_date) \
                   .filterBounds(aoi) \
                   .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
                   .map(maskS2clouds)
    
    # Get the single best image (least cloudy)
    image = collection.sort('CLOUDY_PIXEL_PERCENTAGE').first()

    # Calculate NDVI: (NIR - Red) / (NIR + Red)
    # For Sentinel-2: B8 is NIR, B4 is Red
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # Clip the NDVI image to the exact polygon shape
    ndvi_clipped = ndvi.clip(aoi)

    print("✅ NDVI calculation complete.")

except Exception as e:
    print(f"❌ Error during NDVI calculation: {e}")
    raise

# --- 4. Map Generation ---
print("Generating map...")

# Define visualization parameters for NDVI
# Palette from red (low veg) to green (high veg)
ndvi_vis_params = {
    'min': -0.1,
    'max': 0.8,
    'palette': ['red', 'yellow', 'green']
}

# Get the center of the polygon to center the map
try:
    centroid = aoi.centroid().getInfo()['coordinates']
    centroid.reverse() # Folium expects [lat, lon]
except Exception as e:
    print(f"Warning: Could not get centroid. Using default center. Error: {e}")
    centroid = [12.3915, 77.7737] # Fallback center

# Create a folium map object
m = folium.Map(location=centroid, zoom_start=18)

# Add the EE tile layer to the map
try:
    map_id_dict = ndvi_clipped.getMapId(ndvi_vis_params)
    folium.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name='NDVI',
        overlay=True,
        control=True
    ).add_to(m)

    # Add the polygon outline to the map
    folium.GeoJson(
        data=aoi.getInfo(),
        name='Farm Boundary',
        style_function=lambda x: {'color': 'blue', 'fillOpacity': 0.1, 'weight': 2}
    ).add_to(m)

    # Add a layer control panel
    folium.LayerControl().add_to(m)

    print("✅ Map successfully generated.")
    print("Displaying map below...")

except Exception as e:
    print(f"❌ Error adding tile layer to map: {e}")
    raise

# Display the map in your notebook
m

Authenticating with Google Earth Engine...
✅ Authentication successful for project: wrkfarm-415118
Calculating NDVI...
✅ NDVI calculation complete.
Generating map...
✅ Map successfully generated.
Displaying map below...
